In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# Baseline 04: Mean-by-hour per home (MEMORY-SAFE)
# ============================================================

BASE_DIR = Path("C:/IDEAL_Programming")
DATA_PATH = BASE_DIR / "processed" / "final" / "IDEAL_final_hourly_dataset.csv"
OUT_DIR = BASE_DIR / "processed" / "baselines" / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

# -----------------------------
# Metrics (avoid object arrays)
# -----------------------------
def safe_mape(y, yhat):
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)
    mask = y > 0
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y[mask] - yhat[mask]) / y[mask])) * 100)

def compute_metrics(y, yhat):
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)
    return {
        "MAE": float(np.mean(np.abs(y - yhat))),
        "RMSE": float(np.sqrt(np.mean((y - yhat) ** 2))),
        "MAPE_%": safe_mape(y, yhat),
        "rows_evaluable": int(len(y))
    }

# -----------------------------
# Load ONLY needed columns with dtypes
# -----------------------------
usecols = [HOME_COL, TIME_COL, TARGET_COL]

df = pd.read_csv(
    DATA_PATH,
    usecols=usecols,
    dtype={HOME_COL: "string"}, # keep as pandas string (not python object)
    parse_dates=[TIME_COL],
)

# Force numeric target (avoid object)
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce").astype("float32")
df = df.dropna(subset=[TIME_COL, TARGET_COL]).copy()

# Add hour as small int
df["hour"] = df[TIME_COL].dt.hour.astype("int8")

# Optional: make home_id categorical to reduce memory
df[HOME_COL] = df[HOME_COL].astype("category")

print("Rows:", len(df))
print("Homes:", df[HOME_COL].nunique())

# -----------------------------
# Baseline04: per-home mean by hour
# Build lookup series: index=(home_id, hour) -> mean consumption
# -----------------------------
means = df.groupby([HOME_COL, "hour"], observed=True)[TARGET_COL].mean()

# Map prediction without merge (cheaper)
key = pd.MultiIndex.from_frame(df[[HOME_COL, "hour"]])
df["yhat"] = means.reindex(key).to_numpy(dtype="float32")

df_eval = df.dropna(subset=["yhat"]).copy()

# -----------------------------
# Overall metrics
# -----------------------------
overall = compute_metrics(df_eval[TARGET_COL].to_numpy(), df_eval["yhat"].to_numpy())
overall_df = pd.DataFrame([{"baseline": "mean_by_hour_per_home", **overall}])

print("\n=== Baseline04 (mean-by-hour per home) ===")
print(overall_df)

# -----------------------------
# Per-home metrics (lightweight)
# -----------------------------
# Compute with vectorized groupby aggregations
abs_err = (df_eval[TARGET_COL] - df_eval["yhat"]).abs().astype("float32")
sq_err = ((df_eval[TARGET_COL] - df_eval["yhat"]) ** 2).astype("float32")

perhome = (
    df_eval.assign(abs_err=abs_err, sq_err=sq_err)
    .groupby(HOME_COL, observed=True)
    .agg(
        rows_evaluable=(TARGET_COL, "size"),
        MAE=("abs_err", "mean"),
        RMSE=("sq_err", lambda s: float(np.sqrt(s.mean()))),
    )
    .reset_index()
)

# MAPE per home 
# def home_mape(g):
# return safe_mape(g[TARGET_COL].to_numpy(), g["yhat"].to_numpy())
# perhome["MAPE_%"] = df_eval.groupby(HOME_COL, observed=True).apply(home_mape).values

# -----------------------------
# Save outputs
# -----------------------------
overall_path = OUT_DIR / "baseline04_mean_by_hour_per_home_overall.csv"
perhome_path = OUT_DIR / "baseline04_mean_by_hour_per_home_per_home.csv"

overall_df.to_csv(overall_path, index=False)
perhome.to_csv(perhome_path, index=False)

print("\nSaved:")
print(" -", overall_path)
print(" -", perhome_path)

print("\nDone.")

Rows: 1529350
Homes: 254

=== Baseline04 (mean-by-hour per home) ===
                baseline         MAE        RMSE     MAPE_%  rows_evaluable
0  mean_by_hour_per_home  198.614987  363.365533  75.985988         1529350

Saved:
 - C:\IDEAL_Programming\processed\baselines\results\baseline04_mean_by_hour_per_home_overall.csv
 - C:\IDEAL_Programming\processed\baselines\results\baseline04_mean_by_hour_per_home_per_home.csv

Done.
